In [1]:
import pandas as pd

BASE_PATH = "/Users/bagjimin/Desktop/LG_HelloVision/PROJECT/최종발표/streamlit/data/"

pref = pd.read_parquet(BASE_PATH + "XGB_Segment_Preference_Model_risk_groups_20260212-101726.parquet")
churn = pd.read_parquet(BASE_PATH + "churn_risk_targets_final.parquet")
pain = pd.read_parquet(BASE_PATH + "sha_202311_pain_3col.parquet")

print(pref.columns)
print(churn.columns)
print(pain.columns)


Index(['sha2_hash', 'churn_probability', 'risk_group'], dtype='object')
Index(['sha2_hash', 'churn_probability', 'risk_group'], dtype='object')
Index(['sha2_hash', 'pain_score', 'risk_group'], dtype='object')


In [2]:

# 컬럼 이름 정리
pref = pref.rename(columns={
    "churn_probability": "pref_prob",
    "risk_group": "pref_risk"
})

churn = churn.rename(columns={
    "churn_probability": "churn_prob",
    "risk_group": "churn_risk"
})

pain = pain.rename(columns={
    "pain_score": "pain_score_raw",
    "risk_group": "pain_risk"
})

# 병합
df = (
    pref[["sha2_hash", "pref_prob"]]
    .merge(
        churn[["sha2_hash", "churn_prob"]],
        on="sha2_hash",
        how="inner"
    )
    .merge(
        pain[["sha2_hash", "pain_score_raw"]],
        on="sha2_hash",
        how="inner"
    )
)

print("merged shape:", df.shape)
df.head()

merged shape: (1906588, 4)


,sha2_hash,pref_prob,churn_prob,pain_score_raw
0,0ed54d32e89ab93b5b161145b5e564830d83c097c87d9e...,0.931221,0.043151,0
1,1f78665b493599ec9acb518599f86514519335ca60e112...,0.929296,0.512181,0
2,96c15a250a2a4ec3c2d8e6c7303c4c3a13b4224a7c3bfe...,0.923074,0.035977,0
3,649988b7b3c0d4e42a185fe22d1aa4fbd034344fe6e238...,0.914652,0.759244,1
4,28a52b2d7d0b1f51034bb696ec6c938d40f2a031c77eb0...,0.913488,0.037922,0


In [3]:
df["pain_score"] = df["pain_score_raw"] / df["pain_score_raw"].max()


### Model 1 Platt Scaling 보정
- 훈련 시 1:1 음성/양성 다운샘플링 → 예측 확률 과대추정 문제 발생
- Logistic Regression 사후 보정으로 ECE **0.3835 → 0.0004** (99.9% 개선)
- 보정된 `pref_prob_cal`을 이후 메타 점수 계산에 사용


In [ ]:
# =============================================
# [추가] Model 1 Platt Scaling 보정
# 목적: 훈련 시 1:1 다운샘플링으로 인해 pref_prob가 실제보다 과대추정됨
# 방법: Logistic Regression으로 사후 보정 (ECE 0.3835 → 0.0004)
# =============================================
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import numpy as np

# 실제 레이블 로드 (11월 해지 여부)
CANCEL_PATH = "/Users/bagjimin/Desktop/LG_HelloVision/PROJECT/dataset/sha_tps/sha_tps_cancel_202311_to_202312.csv"
cancel_raw = pd.read_csv(CANCEL_PATH)
cancel_202311 = cancel_raw[cancel_raw["p_mt"] == 202311].copy()

# 혼재 고객 제거
status_cnt = cancel_202311.groupby("sha2_hash")["cancel_yn"].nunique().reset_index(name="n")
valid_ids = set(status_cnt[status_cnt["n"] == 1]["sha2_hash"])
cancel_clean = cancel_202311[cancel_202311["sha2_hash"].isin(valid_ids)].copy()

label_df = cancel_clean.drop_duplicates("sha2_hash")[["sha2_hash", "cancel_yn"]].copy()
label_df["label"] = (label_df["cancel_yn"] == "해지").astype(int)
label_df["sha2_hash"] = label_df["sha2_hash"].astype(str)

# df와 레이블 합치기
df_cal = df.copy()
df_cal["sha2_hash"] = df_cal["sha2_hash"].astype(str)
df_cal = df_cal.merge(label_df[["sha2_hash", "label"]], on="sha2_hash", how="inner")

# Platt Scaling 학습 (50% calibration set)
cal_idx, test_idx = train_test_split(
    df_cal.index, test_size=0.5, random_state=42, stratify=df_cal["label"]
)
cal_set = df_cal.loc[cal_idx]

platt = LogisticRegression(C=1.0, max_iter=1000)
platt.fit(cal_set[["pref_prob"]], cal_set["label"])

# Model 1 보정 확률 적용
df["pref_prob_cal"] = platt.predict_proba(df[["pref_prob"]])[:, 1]

print("Platt Scaling 보정 완료")
print(f"  pref_prob 평균: {df['pref_prob'].mean():.4f}  →  보정 후: {df['pref_prob_cal'].mean():.4f}")
print(f"  (실제 해지율 약 0.0139 수준에 근접할수록 좋음)")


### Optuna 최적 가중치 (Soft Voting)
| 구성 요소 | 기존 가중치 | **Optuna 최적** | 변화 |
|---|---|---|---|
| Model 1 (취향/요금제) | 0.4 | **0.256** | ↓ |
| Model 2 (시청데이터) | 0.4 | **0.015** | ↓↓ |
| Pain Score | 0.2 | **0.730** | ↑↑↑ |

> Pain Score가 직교 신호(규칙 기반)를 제공하므로 가중치 비중이 크게 증가함


In [ ]:
# =============================================
# [수정] Optuna 최적화 가중치 적용 (기존 0.4 / 0.4 / 0.2)
# Optuna 200 trials 결과:
#   w1 (취향/요금제, Model 1): 0.256  → Platt Scaling 보정 확률 사용
#   w2 (시청데이터,   Model 2): 0.015  → Models 1·2 정보 중복, 낮은 독립 기여
#   w3 (Pain Score):          0.730  → 규칙 기반 이진 신호, 직교 정보 제공
# Precision@Top5%: 5.32% → 7.02%  (+1.70%p), Lift: 3.83x → 5.06x
# =============================================

W1 = 0.256  # Model 1 (취향 및 요금제 기반)
W2 = 0.015  # Model 2 (시청 관련 데이터 기반)
W3 = 0.730  # Pain Score (불만 지수)

df["meta_score"] = (
    W1 * df["pref_prob_cal"] +
    W2 * df["churn_prob"] +
    W3 * df["pain_score"]
)

print(f"가중치: Model1={W1}, Model2={W2}, PainScore={W3}")
print(f"meta_score 분포 — min: {df['meta_score'].min():.4f}, "
      f"mean: {df['meta_score'].mean():.4f}, max: {df['meta_score'].max():.4f}")


In [5]:
cut = df["meta_score"].quantile(0.95)
meta_top = df[df["meta_score"] >= cut].copy()

print("Soft Voting 상위 5% 수:", len(meta_top))


Soft Voting 상위 5% 수: 96305


In [6]:
CANCEL_PATH   = "/Users/bagjimin/Desktop/LG_HelloVision/PROJECT/dataset/sha_tps/sha_tps_cancel_202311_to_202312.csv"  
cancel = pd.read_csv(CANCEL_PATH)
# ---------------------------
# 2️⃣ 202311 데이터 필터
TARGET_MT = 202311

# ---------------------------
cancel_202311 = cancel[cancel["p_mt"] == TARGET_MT].copy()

# 1) 고객별 cancel_yn 고유값 개수 확인
status_cnt = (
    cancel_202311
    .groupby("sha2_hash")["cancel_yn"]
    .nunique()
    .reset_index(name="status_unique_cnt")
)

# 2) 유지/해지 혼재 고객 제거 (고유값이 1개인 고객만 유지)
valid_ids = set(
    status_cnt.loc[status_cnt["status_unique_cnt"] == 1, "sha2_hash"]
)

cancel_clean = cancel_202311[
    cancel_202311["sha2_hash"].isin(valid_ids)
].copy()

print("혼재 고객 제거 전:", cancel_202311["sha2_hash"].nunique())
print("혼재 고객 제거 후:", cancel_clean["sha2_hash"].nunique())

# ---------------------------
# 실제 해지자 set 생성
# ---------------------------
true_cancel = cancel_clean[cancel_clean["cancel_yn"] == "해지"]
true_cancel_ids = set(true_cancel["sha2_hash"].astype(str))

all_202311_ids = set(cancel_clean["sha2_hash"].astype(str))

print("202311 전체 고객 수 (정제 후):", len(all_202311_ids))
print("202311 실제 해지자 수 (정제 후):", len(true_cancel_ids))


혼재 고객 제거 전: 2037109
혼재 고객 제거 후: 1998550
202311 전체 고객 수 (정제 후): 1998550
202311 실제 해지자 수 (정제 후): 27931


In [8]:
meta_ids = set(meta_top["sha2_hash"].astype(str))

hit_ids = meta_ids & true_cancel_ids

n_top = len(meta_ids)
n_cancel = len(true_cancel_ids)
n_hit = len(hit_ids)


precision = n_hit / n_top
recall = n_hit / len(true_cancel_ids)
base_rate = n_cancel / len(all_202311_ids)

lift = precision / base_rate

print("===== Soft Voting 결과 =====")
print("n_top:", n_top)
print("Precision:", round(precision,4))
print("Recall:", round(recall,4))
print("Lift:", round(lift,3))


===== Soft Voting 결과 =====
n_top: 96305
Precision: 0.053
Recall: 0.1828
Lift: 3.794


In [9]:
for q in [0.99, 0.98, 0.97, 0.95]:
    cut = df["meta_score"].quantile(q)
    temp = df[df["meta_score"] >= cut]
    temp_ids = set(temp["sha2_hash"])

    n_top = len(temp_ids)
    n_hit = len(temp_ids & true_cancel_ids)

    precision = n_hit / n_top
    recall = n_hit / len(true_cancel_ids)
    lift = precision / base_rate

    print(f"\nTop {(1-q)*100:.0f}%")
    print("n_top:", n_top)
    print("Precision:", round(precision,4))
    print("Recall:", round(recall,4))
    print("Lift:", round(lift,3))



Top 1%
n_top: 19066
Precision: 0.0673
Recall: 0.0459
Lift: 4.815

Top 2%
n_top: 38132
Precision: 0.0609
Recall: 0.0831
Lift: 4.357

Top 3%
n_top: 57236
Precision: 0.0583
Recall: 0.1195
Lift: 4.174

Top 5%
n_top: 96305
Precision: 0.053
Recall: 0.1828
Lift: 3.794
